In [ ]:
import pandas as pd
import numpy as np
import re
import os

In [ ]:
# apartment_monthly_cost_app.csv — all 9 dispositions for the app
# reads from district_stats produced by 1_apartments_cleaner.ipynb
district_stats = pd.read_csv('../data/clean/apartment_district_stats.csv')

keep = ['GARSONIERA', '1+KK', '1+1', '2+KK', '2+1', '3+KK', '3+1', '4+KK', '4+1']
apt_app = district_stats[district_stats['disposition_clean'].isin(keep)][['zone', 'disposition_clean', 'avg_rent']].copy()

apt_app.to_csv('../data/clean/apartment_monthly_cost_app.csv', index=False)
print(apt_app)

In [ ]:
rohlik = pd.read_csv('../data/raw/rohlik_prices.csv')
kosik  = pd.read_csv('../data/raw/kosik_prices.csv')
billa  = pd.read_csv('../data/raw/billa_prices.csv')
lidl   = pd.read_csv('../data/raw/lidl_prices.csv')

In [ ]:
WEEKLY_QTY = {
    'ovesne_vlocky': 500, 'mleko': 2000, 'banany': 1000, 'chleb': 500,
    'maslo': 250, 'vejce': 10, 'spagety': 500, 'pesto': 190, 'ryze': 500,
    'mrazena_zelenina': 400, 'kureci_prsa': 500, 'tofu': 400, 'cibule': 500,
    'cesnek': 80, 'syr_eidam': 200, 'tunak': 370, 'jogurt': 300, 'kava': 250,
    'ovesny_napoj': 1000, 'sojovy_napoj': 1000, 'repkovy_olej': 200,
    'cervena_cocka': 500, 'rajcatova_omacka': 350, 'paprika': 300,
}

BASKET_STANDARD = {
    'ovesne_vlocky', 'mleko', 'banany', 'chleb', 'maslo', 'vejce',
    'spagety', 'pesto', 'ryze', 'mrazena_zelenina', 'kureci_prsa',
    'tofu', 'cibule', 'cesnek', 'syr_eidam', 'tunak', 'jogurt', 'kava',
}

BASKET_VEGAN = {
    'ovesne_vlocky', 'banany', 'chleb', 'spagety', 'ryze',
    'mrazena_zelenina', 'tofu', 'cibule', 'cesnek', 'kava',
    'ovesny_napoj', 'sojovy_napoj', 'repkovy_olej',
    'cervena_cocka', 'rajcatova_omacka', 'paprika',
}

In [ ]:
def parse_qty(text):
    if not text or pd.isna(text):
        return None, None
    t = str(text).lower().strip()
    if m := re.search(r'(\d+[,.]?\d*)\s*ks', t):        return int(float(m.group(1).replace(',','.'))), 'piece'
    if m := re.search(r'(\d+[,.]?\d*)\s*kg', t):        return float(m.group(1).replace(',','.')) * 1000, 'g'
    if m := re.search(r'(\d+[,.]?\d*)\s*g(?!\w)', t):  return float(m.group(1).replace(',','.')), 'g'
    if m := re.search(r'(\d+[,.]?\d*)\s*l(?!\w)', t):  return float(m.group(1).replace(',','.')) * 1000, 'ml'
    if m := re.search(r'(\d+)\s*ml', t):                return int(m.group(1)), 'ml'
    return None, None


def compute_weekly(df):
    def row_cost(row):
        price = row['price_czk']
        if pd.isna(price):
            return np.nan
        target = WEEKLY_QTY.get(row['item'])
        if target is None:
            return np.nan
        pack_qty, _ = parse_qty(row.get('packaging', ''))
        if pack_qty and pack_qty > 0:
            return price / pack_qty * target
        return price
    df = df.copy()
    df['weekly_cost'] = df.apply(row_cost, axis=1)
    df['weekly_cost'] = df['weekly_cost'].where(df['weekly_cost'] <= 500, np.nan)
    return df[['item', 'weekly_cost']]


r = compute_weekly(rohlik).rename(columns={'weekly_cost': 'rohlik'})
k = compute_weekly(kosik).rename(columns={'weekly_cost': 'kosik'})
b = compute_weekly(billa).rename(columns={'weekly_cost': 'billa'})
l = compute_weekly(lidl).rename(columns={'weekly_cost': 'lidl'})

prices = r.merge(k, on='item', how='outer').merge(b, on='item', how='outer').merge(l, on='item', how='outer')

prices.loc[prices['item'] == 'banany', 'billa'] = np.nan

prices['online_avg']   = prices[['rohlik', 'kosik']].mean(axis=1)
prices['physical_avg'] = prices[['billa', 'lidl']].mean(axis=1)

prices

In [ ]:
def basket_cost(df, basket_items):
    sub = df[df['item'].isin(basket_items)]
    weekly_online   = sub['online_avg'].sum() + 60
    weekly_physical = sub['physical_avg'].sum()
    return {
        'weekly_online':    round(weekly_online, 1),
        'weekly_physical':  round(weekly_physical, 1),
        'monthly_online':   round(weekly_online * 4.33, 1),
        'monthly_physical': round(weekly_physical * 4.33, 1),
    }

standard = basket_cost(prices, BASKET_STANDARD)
vegan    = basket_cost(prices, BASKET_VEGAN)

print('Standard:', standard)
print('Vegan:   ', vegan)

In [ ]:
grocery_monthly = pd.DataFrame({
    'basket':  ['standard', 'standard', 'vegan',    'vegan'],
    'store':   ['online',   'physical', 'online',   'physical'],
    'weekly':  [standard['weekly_online'],   standard['weekly_physical'],
                vegan['weekly_online'],       vegan['weekly_physical']],
    'monthly': [standard['monthly_online'],  standard['monthly_physical'],
                vegan['monthly_online'],      vegan['monthly_physical']],
})

grocery_monthly.to_csv('../data/clean/grocery_monthly.csv', index=False)
print('saved')